# Publication notes
See [the project README](../README.md) and [setup instructions](../docs/setup.md). Outputs were cleared for publication; this file has not been rerun during packaging. Notebook 6 is the current integration development reference; supplied-run evidence and limitations are documented in the README.


In [ ]:
import os
from pathlib import Path
import csv
import sys

import numpy as np
import matplotlib.pyplot as plt

from pydrake.all import QuasidynamicLinearPusher

project_dir = Path(os.environ.get("PUSHER_PROJECT", Path.cwd().parent)).expanduser().resolve()
results_dir = project_dir / "results"
input_path = results_dir / "identification_result.npz"

if not input_path.exists():
    raise FileNotFoundError(
        f"Missing {input_path}. "
        "Run the export cell in Notebook 1 first."
    )

with np.load(input_path, allow_pickle=False) as data:
    task_initial_state = data["post_probe_state"].copy()
    MASS_PRIOR = data["mass_prior"].copy()
    learned_mass = float(data["learned_mass"])
    learned_mass_bounds = data["mass_bounds"].copy()

    # Ground truth is used only for plant simulation and the oracle.
    TRUE_MASS = float(data["true_mass"])

    K_A = float(data["k_a"])
    H = float(data["h"])
    W = float(data["w"])

assert task_initial_state.shape == (2,)
assert np.all(np.isfinite(task_initial_state))
assert K_A > 0 and H > 0 and W > 0
assert TRUE_MASS > 0 and learned_mass > 0

print("Python:", sys.executable)
print("Loaded:", input_path)
print("Initial state [pusher, object] [m]:", task_initial_state)
print("Initial mass interval [g]:", 1000 * MASS_PRIOR)
print("Learned mass interval [g]:", 1000 * learned_mass_bounds)
print("Learned mass estimate [g]:", 1000 * learned_mass)

In [ ]:
class DrakePusher:
    """One-step interface to the original unsmoothed Drake pusher."""

    def __init__(self, mass):
        self.system = QuasidynamicLinearPusher(
            k_a=K_A,
            m_o=float(mass),
            w=W,
            dt=H,
            dynamics_smoothing=0.0,
        )

    def step(self, x, u):
        context = self.system.CreateDefaultContext()

        context.SetDiscreteState(np.asarray(x, dtype=float))
        self.system.get_input_port().FixValue(
            context, np.array([float(u)])
        )

        update = self.system.EvalUniquePeriodicDiscreteUpdate(context)
        context.SetDiscreteState(update)

        next_x = context.get_discrete_state_vector().CopyToVector()

        if not np.all(np.isfinite(next_x)):
            raise RuntimeError("Drake returned a nonfinite state.")

        return next_x


def alpha_from_mass(mass):
    return K_A / (K_A + np.asarray(mass) / H**2)


def analytical_step(x, u, mass):
    """Predict the next [pusher position, object position]."""

    object_position = float(x[1])
    excitation = max(0.0, float(u) + W - object_position)

    alpha = float(alpha_from_mass(mass))
    next_object = object_position + alpha * excitation

    if excitation > 0.0:
        next_pusher = next_object - W
    else:
        next_pusher = float(u)

    return np.array([next_pusher, next_object])


print("Drake adapter and analytical model ready.")

In [ ]:
TASK_DISTANCE = 0.005             # 5 mm
TASK_STEPS = 20
MAX_PLANNED_EXCITATION = 0.020    # 20 mm

task_goal = task_initial_state[1] + TASK_DISTANCE

planning_masses = {
    "Before learning": float(np.mean(MASS_PRIOR)),
    "After learning": learned_mass,
    "Oracle": TRUE_MASS,
}

print("Starting object position [m]:", task_initial_state[1])
print("Target object position [m]:", task_goal)
print("Requested motion [mm]:", 1000 * TASK_DISTANCE)
print("Task duration [s]:", TASK_STEPS * H)
print("Desired motion per step [mm]:",
      1000 * TASK_DISTANCE / TASK_STEPS)

for name, mass in planning_masses.items():
    print(
        f"{name}: mass={1000 * mass:.6f} g, "
        f"response factor={float(alpha_from_mass(mass)):.8f}"
    )

In [ ]:
def make_open_loop_plan(initial_state, goal, mass, num_steps):
    initial_state = np.asarray(initial_state, dtype=float).copy()

    desired_increment = (goal - initial_state[1]) / num_steps

    if desired_increment <= 0:
        raise ValueError("This experiment requires forward motion.")

    alpha = float(alpha_from_mass(mass))
    excitation = desired_increment / alpha

    if excitation > MAX_PLANNED_EXCITATION:
        raise ValueError(
            f"Required excitation {excitation:.6f} m exceeds "
            f"{MAX_PLANNED_EXCITATION:.6f} m. "
            "Increase TASK_STEPS."
        )

    nominal_states = [initial_state]
    controls = []

    for _ in range(num_steps):
        nominal_x = nominal_states[-1]

        # Absolute pusher equilibrium-position command.
        u = nominal_x[1] - W + excitation

        next_nominal = analytical_step(nominal_x, u, mass)

        controls.append(u)
        nominal_states.append(next_nominal)

    nominal_states = np.asarray(nominal_states)
    controls = np.asarray(controls)

    if not np.isclose(
        nominal_states[-1, 1], goal, atol=1e-10, rtol=0
    ):
        raise RuntimeError("Nominal plan did not reach its target.")

    return {
        "planning_mass": float(mass),
        "nominal_states": nominal_states,
        "controls": controls,
        "planned_excitation": excitation,
    }


plans = {
    name: make_open_loop_plan(
        task_initial_state,
        task_goal,
        mass,
        TASK_STEPS,
    )
    for name, mass in planning_masses.items()
}

for name, plan in plans.items():
    print(
        f"{name}: "
        f"nominal endpoint={plan['nominal_states'][-1, 1]:.9f} m, "
        f"excitation={1000 * plan['planned_excitation']:.3f} mm"
    )

In [ ]:
def execute_open_loop(initial_state, controls, true_mass):
    plant = DrakePusher(true_mass)

    actual_states = [
        np.asarray(initial_state, dtype=float).copy()
    ]
    actual_excitations = []

    for u in controls:
        current_x = actual_states[-1]

        excitation = max(
            0.0, float(u) + W - current_x[1]
        )
        actual_excitations.append(excitation)

        next_x = plant.step(current_x, float(u))
        actual_states.append(next_x)

    return (
        np.asarray(actual_states),
        np.asarray(actual_excitations),
    )


results = {}

for name, plan in plans.items():
    actual_states, actual_excitations = execute_open_loop(
        task_initial_state,
        plan["controls"],
        TRUE_MASS,
    )

    tracking_error = (
        actual_states[:, 1] - plan["nominal_states"][:, 1]
    )

    results[name] = {
        **plan,
        "actual_states": actual_states,
        "actual_excitations": actual_excitations,
        "tracking_error": tracking_error,
        "final_error": float(actual_states[-1, 1] - task_goal),
        # Exclude the common zero-error initial sample.
        "tracking_rmse": float(
            np.sqrt(np.mean(tracking_error[1:]**2))
        ),
    }

print("Completed all three open-loop executions.")

In [ ]:
print(
    f"{'Method':<18}"
    f"{'Final error [um]':>19}"
    f"{'Tracking RMSE [um]':>22}"
    f"{'Final position [m]':>22}"
)

for name, result in results.items():
    print(
        f"{name:<18}"
        f"{1e6 * result['final_error']:>19.6f}"
        f"{1e6 * result['tracking_rmse']:>22.6f}"
        f"{result['actual_states'][-1, 1]:>22.9f}"
    )

print("\nActual contact-excitation ranges:")

for name, result in results.items():
    a = result["actual_excitations"]
    print(
        f"{name}: {1000 * a.min():.3f} "
        f"to {1000 * a.max():.3f} mm"
    )

before_error = abs(results["Before learning"]["final_error"])
after_error = abs(results["After learning"]["final_error"])

if before_error > 0:
    reduction = 100 * (1 - after_error / before_error)
    print(
        "\nReduction in absolute final error after learning: "
        f"{reduction:.2f}%"
    )

print("\nPositive final error = overshoot.")
print("Negative final error = undershoot.")

In [ ]:
times = np.arange(TASK_STEPS + 1) * H
command_times = np.arange(TASK_STEPS) * H
initial_object = task_initial_state[1]

colors = {
    "Before learning": "tab:blue",
    "After learning": "tab:orange",
    "Oracle": "tab:green",
}

comparison_figure, axes = plt.subplots(
    1, 3, figsize=(16, 4)
)

reference = plans["Oracle"]["nominal_states"][:, 1]

axes[0].plot(
    times,
    1000 * (reference - initial_object),
    "k--",
    label="Planned motion",
)

for name, result in results.items():
    actual_object = result["actual_states"][:, 1]

    axes[0].plot(
        times,
        1000 * (actual_object - initial_object),
        color=colors[name],
        label=name,
    )

    axes[1].plot(
        times,
        1e6 * result["tracking_error"],
        color=colors[name],
        label=name,
    )

    axes[2].step(
        command_times,
        1000 * (
            result["controls"] - (initial_object - W)
        ),
        where="post",
        color=colors[name],
        label=name,
    )

axes[0].set_xlabel("Time [s]")
axes[0].set_ylabel("Object motion from task start [mm]")
axes[0].set_title("Planned and executed motion")

axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("Actual minus planned position [um]")
axes[1].set_title("Tracking error without feedback")

axes[2].set_xlabel("Time [s]")
axes[2].set_ylabel(
    "Command relative to initial contact boundary [mm]"
)
axes[2].set_title("Fixed pusher commands")

for ax in axes:
    ax.grid(True)
    ax.legend()

comparison_figure.tight_layout()
plt.show()

In [ ]:
run_number = 1

while True:
    output_dir = results_dir / f"planning_comparison_{run_number:03d}"
    try:
        output_dir.mkdir(parents=True, exist_ok=False)
        break
    except FileExistsError:
        run_number += 1

comparison_figure.savefig(
    output_dir / "trajectory_comparison.png",
    dpi=200,
    bbox_inches="tight",
)

method_names = list(results)

np.savez(
    output_dir / "planning_comparison.npz",
    method_names=np.asarray(method_names),
    initial_state=task_initial_state,
    task_goal=task_goal,
    task_steps=TASK_STEPS,
    k_a=K_A,
    h=H,
    w=W,
    true_mass=TRUE_MASS,
    planning_masses=np.array([
        results[name]["planning_mass"] for name in method_names
    ]),
    controls=np.stack([
        results[name]["controls"] for name in method_names
    ]),
    nominal_states=np.stack([
        results[name]["nominal_states"] for name in method_names
    ]),
    actual_states=np.stack([
        results[name]["actual_states"] for name in method_names
    ]),
)

with (output_dir / "summary.csv").open(
    "w", newline="", encoding="utf-8"
) as file:
    writer = csv.writer(file)
    writer.writerow([
        "method",
        "planning_mass_kg",
        "final_error_m",
        "tracking_rmse_m",
    ])

    for name, result in results.items():
        writer.writerow([
            name,
            result["planning_mass"],
            result["final_error"],
            result["tracking_rmse"],
        ])

print("Saved results to:", output_dir)